<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/19-efficient-scalable-training.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **高效与可扩展训练** {#efficient-scalable-training}

高效训练研究的是如何把固定的硬件、时间、能源与可靠性预算转化为可信的优化进展。某个 kernel 很快，并不代表整个训练高效：输入停顿、激活显存、梯度通信、重复编译、checkpoint 暂停和失败作业都会增加达到目标结果的时间。**可扩展（scalable）**意味着增加资源能够扩大可行的模型或数据规模，或者缩短完成时间，同时不会造成不可接受的利用率损失或数值漂移。

本章采用一个可复现的贯穿式工作负载：scikit-learn 保存的 [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) 数据副本。它包含 1,797 张标准化的 8×8 图像和十个类别；UCI 数据以 **CC BY 4.0** 许可发布。这里刻意使用一个对任务而言偏深的残差 MLP，使显存、microbatch、计算图捕获、路由、性能分析和 checkpoint 状态在 CPU 上仍然可见。这些实验验证机制，而不宣称 GPU 集群加速比。

![UCI Optical Recognition of Handwritten Digits 工作负载中每个类别的一张样本。](assets/dl19-digits-grid.png){fig-align="center" width="66%" fig-alt="两行网格展示从零到九每个数字类别的一张 8×8 灰度手写图像。"}

*数据来源：Alpaydin 与 Kaynak，[UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49)，CC BY 4.0。图中样本来自 scikit-learn 文档所述的 `load_digits` 副本。*

<details>
<summary><strong>PyTorch：建立共享 Digits 工作负载与确定性划分</strong></summary>

```python
import copy
import io
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1919):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_indices = np.arange(len(digits.data))
train_ids, remaining_ids = train_test_split(
    all_indices, test_size=0.30, stratify=digits.target, random_state=1919
)
val_ids, test_ids = train_test_split(
    remaining_ids, test_size=0.50, stratify=digits.target[remaining_ids], random_state=1919
)

# The fixed 0..16 pixel scale is documented by UCI; no test statistic is fitted.
features = torch.tensor(digits.data / 16.0, dtype=torch.float32)
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(features[train_ids], targets[train_ids])
val_dataset = TensorDataset(features[val_ids], targets[val_ids])
test_dataset = TensorDataset(features[test_ids], targets[test_ids])
loader_generator = torch.Generator().manual_seed(1919)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class ResidualMLPBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.linear1 = nn.Linear(width, 2 * width)
        self.linear2 = nn.Linear(2 * width, width)

    def forward(self, x):
        residual = x
        x = self.linear2(F.gelu(self.linear1(self.norm(x))))
        return residual + 0.25 * x


class DigitMLP(nn.Module):
    def __init__(self, width=128, depth=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(64, width), nn.GELU())
        self.blocks = nn.Sequential(*[ResidualMLPBlock(width) for _ in range(depth)])
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 10))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


def accuracy(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += int((model(x).argmax(1) == y).sum())
            total += len(y)
    return correct / total


seed_everything()
baseline_model = DigitMLP()
baseline_optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(10):
    baseline_model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(baseline_model(x), y)
        baseline_optimizer.zero_grad(); loss.backward(); baseline_optimizer.step()

batch_inputs, batch_targets = next(iter(DataLoader(train_dataset, batch_size=64, shuffle=False)))
parameter_count = sum(parameter.numel() for parameter in baseline_model.parameters())
assert len(set(train_ids) & set(test_ids)) == 0 and batch_inputs.shape == (64, 64)
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "parameters": parameter_count, "test accuracy": round(accuracy(baseline_model, test_loader), 3)})
```

</details>

固定的数据划分、batch、模型定义和已训练基线会从头到尾复用。硬件相关 API 采用带条件保护的生产写法；CPU 模拟只用于声明代数等价性或资源核算，不冒充真实分布式测量。

### **加速器与训练成本模型** {#accelerators-training-cost-model}

加速器之所以有效，是因为稠密线性代数暴露了大量可并行的乘加运算，并能在高速片上存储中重复使用数据。厂商标称的峰值 FLOP/s 只是上限。一个训练 step 还要读取参数与激活、写入中间结果、启动 kernel、同步设备并等待输入。因此，第一项工作是确认真正受限的资源。

对于包含 $F$ 次有效浮点运算、从限制性存储层传输 $Q$ 字节的操作，算术强度为 $I=F/Q$。Roofline 上界为

$$
P_{\mathrm{attainable}}\leq\min(P_{\mathrm{peak}},\;I\,B_{\mathrm{memory}}),
$$

其中，$P_{\mathrm{peak}}$ 是峰值计算吞吐，$B_{\mathrm{memory}}$ 是内存带宽。分布式执行还会加入通信时间。一个简单的非重叠 step 模型为

$$
T_{\mathrm{step}}\approx T_{\mathrm{input}}+T_{\mathrm{compute}}+T_{\mathrm{communication}}+T_{\mathrm{optimizer}}+T_{\mathrm{checkpoint/amortized}}.
$$

![Roofline 模型区分内存带宽受限区与计算受限区。](assets/dl19-roofline.svg){fig-align="center" width="73%" fig-alt="Roofline 图在带宽受限区随算术强度上升，在达到峰值计算上限后转为水平。"}

模型 FLOPs 利用率（MFU）用每秒估算的有效模型 FLOPs 除以硬件峰值 FLOP/s。只有当 FLOP 计数约定、精度、稀疏性和峰值规格一致时，这个指标才有意义。端到端 samples/s 或 tokens/s 仍是运行指标，因为它包含输入与同步成本。

<details>
<summary><strong>Python：估算工作负载的线性层 FLOPs 与 Roofline 上界</strong></summary>

```python
def linear_forward_flops(model, batch_size):
    # A dense [B, in] @ [in, out] multiply is approximately 2*B*in*out FLOPs.
    return sum(
        2 * batch_size * module.in_features * module.out_features
        for module in model.modules() if isinstance(module, nn.Linear)
    )


forward_flops = linear_forward_flops(baseline_model, len(batch_inputs))
training_flops = 3 * forward_flops  # forward + two dominant backward matrix products
parameter_bytes = sum(parameter.numel() * parameter.element_size() for parameter in baseline_model.parameters())
approximate_bytes = parameter_bytes + batch_inputs.numel() * batch_inputs.element_size()
arithmetic_intensity = training_flops / approximate_bytes

# A transparent hypothetical device, not a benchmark of the current CPU.
peak_flops_per_second = 10e12
memory_bytes_per_second = 200e9
roofline = min(peak_flops_per_second, arithmetic_intensity * memory_bytes_per_second)
assert roofline <= peak_flops_per_second
print({"training FLOPs/batch": training_flops, "approx intensity": round(arithmetic_intensity, 1), "hypothetical roofline TFLOP/s": round(roofline / 1e12, 2)})
```

</details>

这个字节估算省略了激活、缓存复用、优化器流量与 workspace，因此只是低保真诊断，并不是性能预测。真实诊断需要 profiler trace 和实测设备计数器。

### **训练内存账本** {#training-memory-ledger}

“模型文件放得下”并不是充分的内存计算。峰值内存是参数、master weight、梯度、优化器状态、保存的激活、临时算子 workspace、通信 bucket 与分配器碎片在某一时刻同时驻留的最大值。

![训练内存账本把静态模型状态、激活和临时分配分开。](assets/dl19-memory-ledger.svg){fig-align="center" width="76%" fig-alt="账本包含参数、梯度、优化器状态、保存的激活以及临时 workspace 或通信 bucket。"}

对于 $P$ 个可训练参数元素，静态账本为

$$
M_{\mathrm{static}}=P(b_w+b_g+b_{m}+b_v+b_{\mathrm{master}}),
$$

其中每个 $b$ 表示每元素字节数，不存在的副本贡献为零。FP32 Adam 通常包含 4 字节权重、4 字节梯度和两个 4 字节 moment：在计入激活前约为 $16P$ 字节。混合精度实现可能在存储低精度模型权重与梯度的同时保留 FP32 master weight，因此“半精度会让显存减半”通常是错误的。

激活内存取决于 batch size、序列或图像形状、层宽、注意力模式以及 autograd 保存了什么。长序列下，它可能超过模型状态。`nvidia-smi` 报告进程级分配；框架计数器则区分存活 tensor、分配器保留块和峰值。框架外 CUDA 分配还可能需要其他工具观察。

<details>
<summary><strong>PyTorch：核算参数状态与为 backward 保存的 tensor</strong></summary>

```python
def tensor_bytes(tensor):
    return tensor.numel() * tensor.element_size()


fp32_parameter_bytes = sum(tensor_bytes(parameter) for parameter in baseline_model.parameters())
fp32_adam_static_bytes = 4 * fp32_parameter_bytes  # weights + gradients + two moments
saved_sizes = []


def pack_hook(tensor):
    saved_sizes.append(tensor_bytes(tensor))
    return tensor


def unpack_hook(tensor):
    return tensor


probe_model = copy.deepcopy(baseline_model).train()
with torch.autograd.graph.saved_tensors_hooks(pack_hook, unpack_hook):
    probe_loss = F.cross_entropy(probe_model(batch_inputs), batch_targets)
    probe_loss.backward()

saved_for_backward_bytes = sum(saved_sizes)
assert fp32_adam_static_bytes > fp32_parameter_bytes and saved_for_backward_bytes > 0
print({"parameter MiB": round(fp32_parameter_bytes / 2**20, 3), "FP32 Adam static MiB": round(fp32_adam_static_bytes / 2**20, 3), "saved tensors MiB": round(saved_for_backward_bytes / 2**20, 3), "saved tensor count": len(saved_sizes)})
```

</details>

Saved-tensor hook 统计的是 autograd 请求的逻辑 tensor，而不是分配器峰值内存；别名和临时 kernel 进一步增加了物理占用的复杂度。该方法适合解释缩放关系，精确诊断则需要 CUDA memory snapshot 与 profiler trace。

### **FP32、FP16、BF16 与 FP8** {#floating-point-formats}

浮点格式把 bit 分配给符号、指数和 fraction。指数 bit 决定动态范围；fraction bit 决定局部精度。FP16 的 fraction bit 多于 BF16，但指数范围窄得多。BF16 与 FP32 一样使用 8 个指数 bit，因此较不容易溢出、通常更容易用于训练，代价是舍入更粗。FP8 也不是单一格式：[FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433) 定义了 E4M3 与 E5M2 的取舍，实际训练方案还需要缩放和更高精度累加。

![FP32、FP16、BF16 与 FP8 为指数和 fraction 分配不同数量的 bit。](assets/dl19-precision-formats.svg){fig-align="center" width="76%" fig-alt="各行比较 FP32、FP16、BF16，以及 FP8 E4M3 或 E5M2 的符号、指数和 fraction 字段。"}

| 格式 | 训练中的典型角色 | 优势 | 主要风险 |
|---|---|---|---|
| FP32 | master state、归约、数值敏感操作 | 范围与精度 | 内存较大，Tensor Core 吞吐较低 |
| FP16 | 矩阵操作与激活 | 紧凑，且在 1 附近精度良好 | 溢出与梯度下溢 |
| BF16 | 矩阵操作与激活 | 接近 FP32 的范围 | fraction bit 较少 |
| FP8 E4M3 | 受支持系统中的前向激活与权重 | 密度更高，精度优于 E5M2 | 范围有限，对缩放敏感 |
| FP8 E5M2 | 梯度或需要更大范围的数值 | FP8 中范围更宽 | 精度非常粗糙 |

<details>
<summary><strong>PyTorch：检查可用格式的范围与舍入误差</strong></summary>

```python
formats = [torch.float32, torch.float16, torch.bfloat16]
if hasattr(torch, "float8_e4m3fn"):
    formats += [torch.float8_e4m3fn, torch.float8_e5m2]

values = torch.tensor([1e-4, 0.1, 1.0, 17.25, 123.0], dtype=torch.float32)
precision_report = {}
for dtype in formats:
    info = torch.finfo(dtype)
    restored = values.to(dtype).float()
    precision_report[str(dtype).replace("torch.", "")] = {
        "tiny": float(info.tiny),
        "max": float(info.max),
        "max abs error": float((values - restored).abs().max()),
    }

assert precision_report["float32"]["max abs error"] <= precision_report["bfloat16"]["max abs error"]
print(precision_report)
```

</details>

对 tensor 执行类型转换并不构成完整的低精度训练方案。矩阵输入、累加、归一化、softmax、优化器状态、梯度归约和通信可能分别使用不同格式。只有硬件支持时，更低精度才会加速计算；否则可能只是增加转换开销。

### **自动混合精度与损失缩放** {#automatic-mixed-precision-loss-scaling}

自动混合精度（AMP）为适合的操作选择低精度，同时为敏感归约或不支持的 kernel 保留更安全的精度。在 PyTorch 中，`autocast` 控制算子 dtype，并不会永久转换模型。[官方 AMP 示例](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) 在 FP16 训练中把 autocast 与 `GradScaler` 配合使用。

![AMP 组合 autocast、损失缩放、反缩放、溢出检查与优化器 step。](assets/dl19-amp-flow.svg){fig-align="center" width="76%" fig-alt="FP32 状态进入 autocast 前向，损失被缩放后反向传播，梯度随后反缩放、检查、裁剪并应用。"}

如果 $g$ 是很小的梯度，$S$ 是缩放因子，对 $S\mathcal L$ 反向传播会产生 $Sg$，把数值移出 FP16 下溢区。在裁剪或更新前，梯度再除以 $S$。动态缩放会在稳定 step 后增大 $S$，在出现非有限梯度后减小 $S$。发生溢出时会跳过优化器 step，因此 scheduler 应跟随真正发生的优化器更新，而不是每次尝试的 batch。

BF16 因为指数范围更宽，通常不需要损失缩放，但仍可能发生溢出或不稳定操作。范数、指数、归约和部分损失通常以 FP32 累加。

<details>
<summary><strong>PyTorch：在共享工作负载上运行一个带保护的 AMP step</strong></summary>

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_model = copy.deepcopy(baseline_model).to(device).train()
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
amp_inputs, amp_targets = batch_inputs.to(device), batch_targets.to(device)
autocast_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

amp_optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type=device.type, dtype=autocast_dtype):
    amp_logits = amp_model(amp_inputs)
    amp_loss = F.cross_entropy(amp_logits, amp_targets)
scaler.scale(amp_loss).backward()
scaler.unscale_(amp_optimizer)  # clipping must see true-scale gradients
gradient_norm = nn.utils.clip_grad_norm_(amp_model.parameters(), max_norm=1.0)
scaler.step(amp_optimizer)
scaler.update()

assert torch.isfinite(amp_loss) and torch.isfinite(gradient_norm)
print({"device": device.type, "autocast dtype": str(autocast_dtype), "logits dtype": str(amp_logits.dtype), "loss": round(float(amp_loss.detach()), 4), "scaler enabled": scaler.is_enabled()})
```

</details>

这里的 CPU BF16 只验证 autocast 控制流程，不验证加速器速度。在 CUDA 上，应检查被跳过的 step、scale 历史、非有限值计数、损失一致性和任务质量，再声称精度迁移成功。

### **梯度累积、激活 Checkpoint 与 Offloading** {#accumulation-checkpointing-offloading}

这些技术缓解不同约束。梯度累积把 global batch 拆成 microbatch，并延迟优化器 step。激活 checkpoint 丢弃选定的前向中间量，在 backward 时重新计算；[原始亚线性内存研究](https://arxiv.org/abs/1604.06174) 形式化了这种计算—内存取舍。Offloading 把状态或激活移动到 CPU 内存或存储设备，并且必须让传输与有效计算重叠，否则会转为带宽受限。

![梯度累积、激活 checkpoint 与 offloading 用不同资源换取内存。](assets/dl19-memory-techniques.svg){fig-align="center" width="76%" fig-alt="三个面板分别展示累积的 microbatch 梯度、重新计算的 checkpoint 区域，以及传输到 CPU 或存储设备的状态。"}

对 $K$ 个等大小 microbatch 和采用 mean reduction 的损失，每个 microbatch 损失都必须除以 $K$，使

$$
\nabla_\theta \mathcal L_{\mathrm{global}}=
\frac{1}{K}\sum_{k=1}^{K}\nabla_\theta\mathcal L_k.
$$

token 数不等时，应按有效 token 总数而不是 microbatch 数量归一化。BatchNorm、dropout mask、梯度裁剪、优化器 schedule 和 DDP 同步都可能破坏严格等价。在 DDP 中，`no_sync()` 用于抑制中间 microbatch 的 all-reduce。

<details>
<summary><strong>PyTorch：验证梯度累积并执行激活 checkpoint 重计算</strong></summary>

```python
from torch.utils.checkpoint import checkpoint_sequential

full_batch_model = copy.deepcopy(baseline_model).train()
accumulated_model = copy.deepcopy(baseline_model).train()
full_loss = F.cross_entropy(full_batch_model(batch_inputs), batch_targets)
full_loss.backward()

microbatch_count = 4
for x_micro, y_micro in zip(batch_inputs.chunk(microbatch_count), batch_targets.chunk(microbatch_count)):
    (F.cross_entropy(accumulated_model(x_micro), y_micro) / microbatch_count).backward()
maximum_gradient_difference = max(
    float((left.grad - right.grad).abs().max())
    for left, right in zip(full_batch_model.parameters(), accumulated_model.parameters())
)

checkpoint_model = copy.deepcopy(baseline_model).train()
block_forward_counts = [0 for _ in checkpoint_model.blocks]
hook_handles = []
for block_index, block in enumerate(checkpoint_model.blocks):
    def count_forward(_module, _inputs, _output, index=block_index):
        block_forward_counts[index] += 1
    hook_handles.append(block.register_forward_hook(count_forward))

def checkpointed_forward(x):
    hidden = checkpoint_model.stem(x)
    hidden = checkpoint_sequential(checkpoint_model.blocks, segments=3, input=hidden, use_reentrant=False)
    return checkpoint_model.head(hidden)

checkpoint_loss = F.cross_entropy(checkpointed_forward(batch_inputs), batch_targets)
checkpoint_loss.backward()
for handle in hook_handles:
    handle.remove()
recomputed_blocks = sum(count > 1 for count in block_forward_counts)
assert maximum_gradient_difference < 1e-5 and torch.isfinite(checkpoint_loss) and recomputed_blocks > 0
print({"accumulation max gradient difference": maximum_gradient_difference, "checkpoint loss": round(float(checkpoint_loss.detach()), 4), "block forward counts": block_forward_counts, "recomputed blocks": recomputed_blocks})
```

</details>

Checkpointed function 在重计算时必须具有确定性；PyTorch 默认保存相关 RNG 状态，但会产生开销。在 checkpointed function 内把 tensor 移动到此前未出现的设备可能破坏等价性。Offloading 还引入另一类正确性问题：过期或过晚预取的状态可能在不易察觉的情况下串行化整个 step。

### **编译与融合 Kernel** {#compilation-fused-kernels}

Eager execution 分别启动各个算子，并物化许多中间 tensor。编译会捕获计算图区域，依据形状与类型 guard 进行专门化，并把兼容操作融合为更少的 kernel。融合减少启动开销和设备内存往返，但不会减少模型的数学 FLOPs。

![编译融合兼容算子，而 graph break 与 guard 变化会建立新区域或触发重编译。](assets/dl19-compile-fusion.svg){fig-align="center" width="75%" fig-alt="Bias、GELU 与 dropout kernel 变成一个融合区域；graph break 切分区域，guard 变化触发重编译。"}

`torch.compile` 通过 guard 保留 Python 语义，并可能编译多个变体。动态控制流、不支持的操作、Python 副作用、变化的形状和标量提取都可能造成 graph break 或重编译。首次编译成本必须得到摊销；短作业反而可能变慢。[PyTorch profiler 指南](https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/torch.compiler_profiling_torch_compile.html) 建议检查 compiled region 和 graph break，而不是因为代码能运行就假定编译成功。

融合优化器、融合归一化与融合注意力是针对常见模式设计的库 kernel。除非 profile 证明缺失 kernel 占主导，并且维护成本合理，否则应优先使用成熟实现，而不是手写 CUDA。

<details>
<summary><strong>PyTorch：验证图捕获，同时不把 CPU eager backend 冒充速度 benchmark</strong></summary>

```python
compile_probe = copy.deepcopy(baseline_model).eval()
eager_output = compile_probe(batch_inputs)
compiled_probe = torch.compile(compile_probe, backend="eager", fullgraph=True)
compiled_output = compiled_probe(batch_inputs)

# FX exposes operator structure; backend='eager' checks capture but intentionally performs no fusion.
fx_graph = torch.fx.symbolic_trace(copy.deepcopy(baseline_model).eval())
call_nodes = [node for node in fx_graph.graph.nodes if node.op in {"call_module", "call_function", "call_method"}]
assert torch.allclose(eager_output, compiled_output, atol=1e-6)
print({"captured output parity": True, "FX call nodes": len(call_nodes), "backend": "eager (capture validation only)"})
```

</details>

生产 benchmark 使用真实优化 backend、warm-up、计时前后同步、代表性形状和质量检查。更少的 kernel 仍可能变慢，例如融合增加寄存器压力、降低 occupancy 或反复触发重编译。

### **数据并行与 DistributedDataParallel** {#data-parallelism-ddp}

数据并行在每个 rank 上放置完整模型副本，划分 global batch，计算局部梯度，再对梯度求平均。如果 rank $r$ 处理 $B_r$ 个等量样本，并且损失采用 mean reduction，同步 DDP 计算

$$
g=\frac{1}{W}\sum_{r=1}^{W}g_r,
$$

其中 $W$ 为 world size。局部 batch 或 token 数量不等时需要加权归约。PyTorch [DistributedDataParallel](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html) 同步梯度，但**不会**自动切分输入；必须由 sampler 或数据服务为每个 rank 分配局部样本。

![DDP 副本处理不同 batch shard，并对 bucket 化梯度求平均。](assets/dl19-ddp-allreduce.svg){fig-align="center" width="76%" fig-alt="三个 rank 处理各自数据 shard，把梯度发送到 bucket 化 all-reduce，并接收平均后的梯度。"}

DDP 注册 autograd hook，并把梯度组织为 bucket，使 all-reduce 与后续 backward 计算重叠。Bucket 大小在启动延迟与重叠之间取舍。`find_unused_parameters=True`、分歧控制流、straggler 和大量微小层都会降低效率。只有每个 rank 的计算足以摊销同步时，DDP 才能提高吞吐。

<details>
<summary><strong>PyTorch：模拟两个 DDP rank 并验证平均梯度等价性</strong></summary>

```python
reference_model = copy.deepcopy(baseline_model).train()
reference_loss = F.cross_entropy(reference_model(batch_inputs), batch_targets)
reference_loss.backward()

rank_models = [copy.deepcopy(baseline_model).train() for _ in range(2)]
for rank_model, x_rank, y_rank in zip(rank_models, batch_inputs.chunk(2), batch_targets.chunk(2)):
    F.cross_entropy(rank_model(x_rank), y_rank).backward()

maximum_allreduce_difference = 0.0
for parameter_index, reference_parameter in enumerate(reference_model.parameters()):
    averaged = torch.stack([
        list(rank_model.parameters())[parameter_index].grad for rank_model in rank_models
    ]).mean(0)
    maximum_allreduce_difference = max(
        maximum_allreduce_difference, float((averaged - reference_parameter.grad).abs().max())
    )

assert maximum_allreduce_difference < 1e-5
print({"world size": 2, "max gradient difference": maximum_allreduce_difference, "global batch": len(batch_inputs)})
```

</details>

该模拟只验证等大小 shard 的梯度代数，不测量 collective 延迟或重叠。真实运行还需要确认初始化一致、控制流确定、设备为 rank-local、sampler epoch 正确，以及普通非分片 checkpoint 等副作用只由 rank 0 执行。

### **FSDP 与 ZeRO** {#fsdp-zero}

DDP 复制参数、梯度与优化器状态，因此增加设备并不会降低每 rank 的模型状态内存。[ZeRO](https://arxiv.org/abs/1910.02054) 在 data-parallel group 中划分冗余状态。ZeRO Stage 1 划分优化器状态，Stage 2 进一步划分梯度，Stage 3 再划分参数。PyTorch [FSDP `FULL_SHARD`](https://docs.pytorch.org/docs/stable/fsdp.html) 类似地在计算前 all-gather 参数 shard，并在之后 reduce-scatter 梯度。

![ZeRO 各阶段逐步划分优化器状态、梯度与参数。](assets/dl19-zero-stages.svg){fig-align="center" width="76%" fig-alt="表格显示 DDP 复制全部模型状态，ZeRO Stage 1 划分优化器状态，Stage 2 进一步划分梯度，Stage 3 或 FSDP 划分三者。"}

假设混合精度 Adam 使用 2 字节模型权重、2 字节梯度、4 字节 master copy 和 8 字节 moment，则简化后的每 rank 账本为

$$
M_{\mathrm{DDP}}\approx16P,\quad
M_{Z1}\approx4P+\frac{12P}{W},\quad
M_{Z2}\approx2P+\frac{14P}{W},\quad
M_{Z3}\approx\frac{16P}{W}.
$$

这里排除了激活、buffer、临时 all-gather 参数、通信 bucket、碎片和 checkpoint staging。Sharding 降低常驻内存，却增加 collective，并使 wrapping 粒度变得重要：过小 FSDP unit 会产生大量 collective；过大 unit 会形成高内存峰值和较弱重叠。

<details>
<summary><strong>Python：计算分片状态内存并重建一个参数 shard</strong></summary>

```python
def model_state_bytes_per_rank(parameters, world_size):
    return {
        "DDP": 16 * parameters,
        "ZeRO-1": 4 * parameters + 12 * parameters / world_size,
        "ZeRO-2": 2 * parameters + 14 * parameters / world_size,
        "ZeRO-3/FSDP": 16 * parameters / world_size,
    }


world_size = 4
actual_ledger = model_state_bytes_per_rank(parameter_count, world_size)
seven_billion_ledger_gib = {
    name: value / 2**30 for name, value in model_state_bytes_per_rank(7_000_000_000, world_size).items()
}

flat_weight = baseline_model.stem[0].weight.detach().flatten()
shards = list(torch.tensor_split(flat_weight, world_size))
reconstructed = torch.cat(shards)
assert torch.equal(flat_weight, reconstructed) and actual_ledger["ZeRO-3/FSDP"] < actual_ledger["DDP"]
print({"actual model MiB/rank": {k: round(v / 2**20, 3) for k, v in actual_ledger.items()}, "7B model GiB/rank (state only)": {k: round(v, 2) for k, v in seven_billion_ledger_gib.items()}})
```

</details>

Full-state checkpoint 可能临时收集整个模型并造成 rank 0 OOM。Sharded state dictionary 和 distributed checkpoint 能避免这个峰值，并支持在不同 world size 下加载时重新分片。

### **张量并行与序列并行** {#tensor-sequence-parallelism}

张量并行划分单个层的计算。对于 $Y=XW^\top+b$，column parallelism 切分 $W$ 的行（输出特征），独立计算输出切片后拼接。Row parallelism 则切分输入特征和 $W$ 的列，计算局部输出后通过 all-reduce 求和。

![Column tensor parallelism 划分输出特征并拼接局部矩阵乘结果。](assets/dl19-tensor-parallel.svg){fig-align="center" width="74%" fig-alt="输入矩阵分别乘以两个输出列权重 shard，局部输出随后拼接为完整结果。"}

[Megatron-LM](https://arxiv.org/abs/1909.08053) 组织 Transformer 的注意力与 MLP projection，使 collective 只出现在受控边界。张量并行降低层内每 rank 的参数与激活驻留量，但每层都引入延迟敏感的通信，因此更适合节点内的高带宽互连。

序列并行沿序列维度划分激活，用于那些不必在每个 tensor-parallel rank 上保留完整序列的操作，例如部分归一化和 dropout 区域。它减少重复激活内存；它不同于 context parallelism，后者划分注意力上下文，并且必须交换 key/value 信息或局部注意力统计量。

<details>
<summary><strong>PyTorch：用 column 与 row partition 复现一个已训练线性层</strong></summary>

```python
linear = baseline_model.stem[0]
x = batch_inputs
full_output = F.linear(x, linear.weight, linear.bias)

# Column parallel: split output rows, compute locally, concatenate features.
weight_output_shards = linear.weight.chunk(2, dim=0)
bias_shards = linear.bias.chunk(2, dim=0)
column_output = torch.cat([
    F.linear(x, weight_shard, bias_shard)
    for weight_shard, bias_shard in zip(weight_output_shards, bias_shards)
], dim=-1)

# Row parallel: split input features and weight columns, sum partial outputs, add bias once.
input_shards = x.chunk(2, dim=-1)
weight_input_shards = linear.weight.chunk(2, dim=1)
row_output = sum(F.linear(input_shard, weight_shard, None) for input_shard, weight_shard in zip(input_shards, weight_input_shards)) + linear.bias

assert torch.allclose(full_output, column_output, atol=1e-6)
assert torch.allclose(full_output, row_output, atol=1e-6)
print({"full shape": tuple(full_output.shape), "column local shape": tuple(column_output[:, : column_output.shape[1] // 2].shape), "row collective": "sum/all-reduce"})
```

</details>

这些局部 tensor 操作证明了划分等价性，但没有模拟 collective 顺序、异步重叠、拓扑或归约顺序变化带来的数值差异。

### **流水线并行与上下文并行** {#pipeline-context-parallelism}

流水线并行把连续模型 stage 分配到不同设备，并把一个 batch 拆为 $m$ 个 microbatch。一个 stage 处理较晚 microbatch 时，另一个 stage 可以处理较早 microbatch。在简单的“全部 forward 后全部 backward”GPipe 调度中，如果有 $p$ 个 stage，理想化 forward 利用率为

$$
\eta_{\mathrm{pipeline}}\approx\frac{m}{m+p-1}.
$$

![流水线先填充、随后让 microbatch 跨 stage 重叠，最后排空。](assets/dl19-pipeline-context.svg){fig-align="center" width="75%" fig-alt="三个模型 stage 在错开的时间线上处理四个 microbatch，并形成填充与排空 bubble。"}

增大 $m$ 会缩小 bubble 比例，但同时减小 microbatch，并增加调度开销。Stage 划分要平衡计算时间与激活传输，而不是简单平衡层数。Interleaved 和 1F1B 调度可以减少空闲时间或激活驻留，却使依赖关系更复杂。[GPipe](https://arxiv.org/abs/1811.06965) 是 batch-splitting pipeline 的经典参考。

上下文并行解决单个设备无法容纳超长序列注意力激活的问题。它在 rank 间划分 token，并交换 key/value block 或局部 softmax 统计量。正确的分布式 softmax 需要全局一致的最大值与归一化和；只关注局部 token 会直接改变模型。

<details>
<summary><strong>PyTorch：把 Digits MLP 划分为 stage，并计算流水线 bubble</strong></summary>

```python
baseline_model.eval()
block_list = list(baseline_model.blocks.children())
stage_0 = nn.Sequential(baseline_model.stem, *block_list[:3])
stage_1 = nn.Sequential(*block_list[3:], baseline_model.head)
staged_output = stage_1(stage_0(batch_inputs))
direct_output = baseline_model(batch_inputs)

pipeline_efficiency = {
    microbatches: microbatches / (microbatches + 3 - 1)
    for microbatches in (1, 2, 4, 8, 16)
}
assert torch.allclose(staged_output, direct_output, atol=1e-6)
print({"stage boundary": tuple(stage_0(batch_inputs).shape), "three-stage idealized efficiency": {k: round(v, 3) for k, v in pipeline_efficiency.items()}})
```

</details>

代码验证的是层划分，但执行仍然串行。真实流水线的正确性还要求 microbatch 损失归一化一致、正确处理 tied weight、重计算具有确定性，以及各 stage 的优化器语义匹配。

### **专家并行与混合专家模型** {#expert-parallelism-moe}

混合专家（MoE）层包含许多带参数的 expert，但每个 token 只路由到 top-$k$ 个。这能在不把每个参数应用到每个 token 的情况下增加参数容量。专家并行把不同 expert 放到不同 rank；token state 经 all-to-all collective 交换、处理后，再返回原来的序列位置。

![MoE router 把 token 分派给受容量限制的 expert，再组合输出。](assets/dl19-moe-routing.svg){fig-align="center" width="74%" fig-alt="Token state 进入 top-k router，被发送到不同 device group 上的 expert，再返回加权组合阶段。"}

令 $p_{te}$ 为 token $t$ 选择 expert $e$ 的 router 概率。Top-1 routing 选择 $e_t=\arg\max_e p_{te}$。容量因子 $c$ 通常把每个 expert 限制在约 $\lceil cT/E\rceil$ 个 token。偏斜会产生 hot expert、被丢弃或改道的 token，以及 rank straggler。辅助负载均衡损失同时鼓励路由概率与实际 assignment 分散到多个 expert。

[Switch Transformer](https://arxiv.org/abs/2101.03961) 把路由简化为每 token 一个 expert，但稀疏计算并不等于通信便宜。Expert placement、token permutation、容量、拓扑与数值稳定性共同决定真实吞吐。

<details>
<summary><strong>PyTorch：把共享工作负载表示路由到有容量限制的 expert</strong></summary>

```python
seed_everything(1926)
with torch.no_grad():
    token_states = baseline_model.stem(batch_inputs)
expert_count = 4
router = nn.Linear(token_states.shape[-1], expert_count, bias=False)
experts = nn.ModuleList([
    nn.Sequential(nn.Linear(128, 192), nn.GELU(), nn.Linear(192, 128))
    for _ in range(expert_count)
])
routing_probabilities = router(token_states).softmax(-1)
assignments = routing_probabilities.argmax(-1)
capacity = math.ceil(1.25 * len(token_states) / expert_count)
moe_output = token_states.clone()
expert_loads, overflow = [], 0
for expert_id, expert in enumerate(experts):
    token_ids = (assignments == expert_id).nonzero(as_tuple=False).squeeze(1)
    expert_loads.append(len(token_ids))
    accepted, rejected = token_ids[:capacity], token_ids[capacity:]
    if len(accepted):
        moe_output[accepted] = expert(token_states[accepted])
    overflow += len(rejected)  # fallback keeps the residual representation in this demonstration

load_fraction = torch.bincount(assignments, minlength=expert_count).float() / len(assignments)
assert moe_output.shape == token_states.shape and sum(expert_loads) == len(token_states)
print({"expert loads": expert_loads, "capacity": capacity, "overflow": overflow, "load coefficient of variation": round(float(load_fraction.std() / load_fraction.mean()), 3)})
```

</details>

这个未训练 router 刻意暴露负载失衡。生产 MoE 会联合训练路由，加入加权 top-$k$ 组合与辅助损失，并同时测量每 expert token 数和 all-to-all 时间，而不只报告参数数量。

### **分布式数据管线** {#distributed-data-pipelines}

如果数据管线解码、增强或传输太慢，再强的加速器也无法补救。完整路径包括存储布局、sharding、shuffle、CPU worker、transform、collation、pinned memory、host-to-device 传输与 device prefetch。每个 rank 必须收到互不重复且统计有效的样本流，并在相容时间到达 collective。

![分布式输入管线让存储读取、CPU transform、pinned buffer、传输与设备计算重叠。](assets/dl19-data-pipeline.svg){fig-align="center" width="75%" fig-alt="存储依次连接 CPU worker、pinned batch buffer、异步设备传输和计算，并让下一 batch 的准备与当前计算重叠。"}

PyTorch [数据加载文档](https://docs.pytorch.org/docs/stable/data.html) 区分 map-style 与 iterable dataset。`DistributedSampler` 划分索引，但每个 epoch 必须调用 `set_epoch(epoch)` 才会产生新的确定性 shuffle。数据集大小不能被 world size 整除时，padding 会复制样本；`drop_last=True` 则丢弃尾部。Iterable dataset 需要显式按 rank 和 worker 分片，避免每个进程读取相同数据流。

Pinned host memory 与 `non_blocking=True` 结合时允许异步 DMA 到 CUDA，但 pinning 过多内存会伤害主机。Worker 数、prefetch 深度和 persistent worker 应依据实测队列停顿调节，而不是盲目取最大值。

<details>
<summary><strong>PyTorch：检查 rank-local 采样与随 epoch 变化的 shuffle</strong></summary>

```python
from torch.utils.data.distributed import DistributedSampler

sampler_indices = {}
for rank in range(4):
    sampler = DistributedSampler(
        train_dataset, num_replicas=4, rank=rank, shuffle=True, seed=1919, drop_last=True
    )
    sampler.set_epoch(0)
    sampler_indices[rank] = list(iter(sampler))

rank_sets = [set(indices) for indices in sampler_indices.values()]
pairwise_overlap = max(len(rank_sets[left] & rank_sets[right]) for left in range(4) for right in range(left + 1, 4))
covered = len(set().union(*rank_sets))

epoch_probe = DistributedSampler(train_dataset, num_replicas=4, rank=0, shuffle=True, seed=1919, drop_last=True)
epoch_probe.set_epoch(0); epoch_zero = list(iter(epoch_probe))
epoch_probe.set_epoch(1); epoch_one = list(iter(epoch_probe))
assert pairwise_overlap == 0 and epoch_zero != epoch_one
print({"examples/rank": len(epoch_zero), "covered": covered, "dropped tail": len(train_dataset) - covered, "rank overlap": pairwise_overlap})
```

</details>

对序列语料而言，相同样本数仍可能造成严重 token 数不平衡。长度感知 packing 能改善利用率，却可能改变样本顺序和梯度统计；精确 packing 与恢复游标都应记录在实验产物中。

### **性能分析、通信与瓶颈诊断** {#profiling-communication-bottlenecks}

优化应从代表性 steady-state step 的时间线开始。Profile 能区分输入空隙、host 启动开销、device kernel、内存分配、通信、优化器工作、graph break 和 checkpoint 暂停。端到端吞吐说明运行**确实慢**，trace 则帮助解释**为什么慢**。

![一个 step 时间线分离 CPU 启动、加速器 kernel、网络 collective 与空闲间隙。](assets/dl19-profiler-timeline.svg){fig-align="center" width="76%" fig-alt="CPU、GPU 与网络轨道展示输入启动、forward、backward、梯度 all-reduce、优化器工作和空闲间隙。"}

对包含 $n$ 字节的 collective 消息，alpha-beta 模型写为 $T\approx\alpha N_{\mathrm{messages}}+\beta n$，其中 $\alpha$ 表示延迟，$\beta$ 表示带宽倒数。小 bucket 会反复支付延迟；巨大 bucket 会延后重叠。同步 step 时间由最慢 rank 决定，因此 per-rank trace 与 straggler percentile 比平均 trace 更重要。

[PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html) 可以记录算子时间、形状、内存、调用栈与设备活动。Warm-up 与 active window 用于排除初始化噪声；分布式 trace 需要对齐时钟并标明 rank。

<details>
<summary><strong>PyTorch：对共享模型的一个完整 CPU 训练 step 进行 profile</strong></summary>

```python
from torch.profiler import ProfilerActivity, profile

profiled_model = copy.deepcopy(baseline_model).train()
profiled_optimizer = torch.optim.SGD(profiled_model.parameters(), lr=1e-3)
with profile(activities=[ProfilerActivity.CPU], record_shapes=True, profile_memory=True, acc_events=True) as trace:
    with torch.profiler.record_function("digit_training_step"):
        profiled_loss = F.cross_entropy(profiled_model(batch_inputs), batch_targets)
        profiled_optimizer.zero_grad(set_to_none=True)
        profiled_loss.backward()
        profiled_optimizer.step()

top_events = trace.key_averages().table(sort_by="self_cpu_time_total", row_limit=5)
assert "Self CPU" in top_events and torch.isfinite(profiled_loss)
print(top_events)
```

</details>

单 step CPU trace 展示的是工具用法，而不是稳定排名。生产 profiling 会丢弃编译和分配器 warm-up，采样多个 iteration，在手动计时前后同步，并把框架事件与设备、网络计数器关联起来。

### **Checkpoint、容错与可复现性** {#checkpointing-fault-tolerance-reproducibility}

可恢复 checkpoint 包含的不只是模型权重。精确继续训练可能需要优化器 moment、scheduler、AMP scaler、global step、epoch 与数据游标、每个 rank 的 RNG 状态、模型配置、tokenizer 或数据版本，以及软件 commit。只保存权重得到的是具有不同优化动力学的 warm restart。

![可靠 checkpoint 在版本化 manifest 下组合模型、优化器、schedule、RNG 与数据游标状态。](assets/dl19-checkpoint-state.svg){fig-align="center" width="76%" fig-alt="模型、优化器、scheduler 与 scaler、随机状态和数据游标共同输入一个版本化且原子写入的 checkpoint manifest。"}

Checkpoint 间隔在写入开销与预期丢失工作之间取舍。如果故障独立、平均故障间隔为 $M$、一次 checkpoint 成本为 $C$，经典近似会选择数量级为 $\sqrt{2CM}$ 的间隔，再根据恢复成本和相关故障调整。异步保存只有在 staging memory 与存储带宽不干扰训练时，才能真正减少暂停。

分片模型应避免在一个 rank 上聚集完整 checkpoint。PyTorch [Distributed Checkpoint](https://docs.pytorch.org/tutorials/recipes/distributed_checkpoint_recipe.html) 并行写入 rank-local shard，并能在另一种拓扑下加载时重新分片。原子 manifest、checksum、保留策略和定期 restore 测试都属于正确性要求。

<details>
<summary><strong>PyTorch：序列化并恢复完整的内存训练状态</strong></summary>

```python
checkpoint_model = copy.deepcopy(baseline_model).train()
checkpoint_optimizer = torch.optim.AdamW(checkpoint_model.parameters(), lr=1e-3)
checkpoint_loss = F.cross_entropy(checkpoint_model(batch_inputs), batch_targets)
checkpoint_optimizer.zero_grad(); checkpoint_loss.backward(); checkpoint_optimizer.step()
checkpoint_model.eval()
with torch.no_grad():
    checkpoint_logits = checkpoint_model(batch_inputs).clone()

state = {
    "model": checkpoint_model.state_dict(),
    "optimizer": checkpoint_optimizer.state_dict(),
    "step": 1,
    "torch_rng": torch.get_rng_state(),
    "numpy_rng": np.random.get_state(),
    "python_rng": random.getstate(),
    "loader_generator": loader_generator.get_state(),
    "split_ids": {"train": train_ids, "validation": val_ids, "test": test_ids},
}
buffer = io.BytesIO()
torch.save(state, buffer)
buffer.seek(0)
restored_state = torch.load(buffer, weights_only=False)

restored_model = DigitMLP()
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=1e-3)
restored_model.load_state_dict(restored_state["model"])
restored_optimizer.load_state_dict(restored_state["optimizer"])
torch.set_rng_state(restored_state["torch_rng"])
np.random.set_state(restored_state["numpy_rng"])
random.setstate(restored_state["python_rng"])
loader_generator.set_state(restored_state["loader_generator"])
restored_model.eval()

with torch.no_grad():
    restored_logits = restored_model(batch_inputs)
assert torch.equal(checkpoint_logits, restored_logits) and restored_state["step"] == 1
print({"checkpoint KiB": round(buffer.getbuffer().nbytes / 2**10, 1), "optimizer state entries": len(restored_state["optimizer"]["state"]), "exact output restore": True})
```

</details>

可复现性存在不同层级：bitwise replay、统计等价训练与最终质量可比是不同承诺。Collective 归约顺序、kernel 选择、异步执行、worker 调度和硬件差异都可能在保存全部 seed 后仍阻止 bitwise 一致。应报告真正测试过的层级。

### **章节对比与总结** {#chapter-comparison-summary}

| 技术 | 主要解决的瓶颈 | 核心代价 | 验证信号 |
|---|---|---|---|
| 混合精度 | Tensor 计算与存储 | 数值范围与舍入 | 损失一致性、溢出率、任务质量 |
| 梯度累积 | Batch 激活内存 | 更多 microstep 和延迟更新 | 梯度等价与 samples/s |
| 激活 checkpoint | 保存的激活 | 重计算 | 峰值内存与 step 时间 |
| Offloading | 设备内存容量 | 传输带宽与延迟 | 重叠程度与停顿时间 |
| 编译/融合 | 启动开销与内存流量 | 编译成本与计算图专门化 | steady-state trace 与重编译次数 |
| DDP | 数据吞吐 | 重复模型状态与 all-reduce | 扩展效率与 straggler |
| FSDP/ZeRO | 重复模型状态 | all-gather/reduce-scatter 流量 | 峰值内存与通信重叠 |
| 张量并行 | 层参数与激活容量 | 每层 collective | 拓扑感知吞吐 |
| 流水线并行 | 模型深度容量 | Bubble 与 stage 不平衡 | Stage 利用率与激活驻留 |
| 上下文并行 | 长序列激活容量 | 分布式注意力通信 | 精确性与序列吞吐 |
| 专家并行 | 稀疏模型容量 | 路由不平衡与 all-to-all | 每 expert 负载与 token 吞吐 |

实用的优化顺序是：

1. 固定正确的单设备基线、数据划分、指标和数值容差。
2. 测量端到端吞吐、峰值内存与代表性 steady-state trace。
3. 在硬件支持时使用 AMP 和高效 kernel，并验证质量与溢出行为。
4. 只针对真正占主导的内存类别应用累积或 checkpoint。
5. 在完整模型仍能放入单设备、all-reduce 能够重叠时使用 DDP 扩展。
6. 当重复模型状态成为容量限制时引入 FSDP/ZeRO。
7. 根据无法继续容纳的具体维度增加张量、流水线、上下文或专家并行。
8. 每次改变拓扑后重新平衡数据、计算与通信。
9. 保存完整且版本化的状态，并在长作业前进行 restore 演练。
10. 同时报告 time-to-quality、利用率、内存、能源或成本假设、故障恢复与数值一致性。

可扩展性是系统属性，而不是一个 API 开关。精度、内存生命周期、计算图结构、collective 拓扑、数据顺序和故障恢复相互作用；改善一个孤立指标可能反而增加 time-to-result。下一章将沿着已训练模型进入高效推理与部署，此时延迟、batching、缓存、量化和服务级目标会取代 backward 约束。